### Creat agent with Agent state


#### Agent State is the data an agent maintains while it executes.


In [ ]:
from langchain.agents import AgentState

messages: list[BaseMessage]

In [ ]:

from typing import Any

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import after_model


# 1. Define custom agent state
class MyState(AgentState):
    user_id: str
    call_count: int


# 2. Update state after every model response
@after_model
def increment_call_count(
    state: MyState,
    runtime,
) -> dict[str, Any]:
    current_count = state.get("call_count", 0)

    return {
        "call_count": current_count + 1
    }


# 3. Create the agent
agent = create_agent(
    model="openai:gpt-5.5",
    tools=[],
    state_schema=MyState,
    middleware=[increment_call_count],
)


# 4. Initialize state and invoke the agent
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain caching in one sentence.",
            }
        ],
        "user_id": "user-123",
        "call_count": 0,
    }
)


# 5. Read the updated state
print("User ID:", result["user_id"])
print("Call count:", result["call_count"])
print("AI response:", result["messages"][-1].content)


#### messages field grows as a LangChain agent processes a conversation

In [ ]:
{
  "messages": [
    {
      "type": "human",
      "content": "What is caching?",
      "id": "msg-human-1"
    },
    {
      "type": "ai",
      "content": "Caching stores frequently accessed data for faster retrieval.",
      "id": "msg-ai-1"
    },
    {
      "type": "human",
      "content": "Give me a real-world example.",
      "id": "msg-human-2"
    },
    {
      "type": "ai",
      "content": "A browser caches images so they load faster when you revisit a website.",
      "id": "msg-ai-2"
    }
  ]
}

### Have not declared user_id in MyState

In [ ]:

from typing import Any

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import after_model


# 1. Define custom agent state
class MyState(AgentState):
    call_count: int


# 2. Update state after every model response
@after_model
def increment_call_count(
    state: MyState,
    runtime,
) -> dict[str, Any]:
    current_count = state.get("call_count", 0)

    return {
        "call_count": current_count + 1
    }


# 3. Create the agent
agent = create_agent(
    model="openai:gpt-5.5",
    tools=[],
    state_schema=MyState,
    middleware=[increment_call_count],
)


# 4. Initialize state and invoke the agent
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain caching in one sentence.",
            }
        ],
        "user_id": "user-123",
        "call_count": 0,
    }
)


# 5. Read the updated state
print("User ID:", result["user_id"])
print("Call count:", result["call_count"])
print("AI response:", result["messages"][-1].content)

### Return a key that isn't in MyState

In [ ]:
from typing import Any

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import after_model


# 1. State does NOT declare user_id
class MyState(AgentState):
    call_count: int


# 2. Return one declared field and one undeclared field
@after_model
def update_state(
    state: MyState,
    runtime,
) -> dict[str, Any]:
    return {
        "call_count": state.get("call_count", 0) + 1,
        "user_id": "user-123",
    }


# 3. Create agent
agent = create_agent(
    model="openai:gpt-5.5",
    tools=[],
    state_schema=MyState,
    middleware=[update_state],
)


# 4. Invoke agent
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Explain caching briefly."}
    ],
    "call_count": 0,
})


# 5. Inspect the result
print("Call count:", result["call_count"])
print("Available keys:", result.keys())

# This may raise KeyError because user_id is not a declared state field:
print("User ID:", result["user_id"])